In [ ]:
import os
from dotenv import load_dotenv
import matplotlib.pyplot as plt
import pandas as pd
import panel as pn
import psycopg2 as pg
from sqlalchemy import URL, create_engine, text

# 1. Carrega variáveis do arquivo python/.env
ENV_PATH = os.path.join(os.getcwd(), ".env")
if not os.path.exists(ENV_PATH):
    ENV_PATH = os.path.join(os.getcwd(), "python", ".env")
load_dotenv(dotenv_path=ENV_PATH)

DB_HOST = os.getenv("DB_HOST")
DB_PORT = int(os.getenv("DB_PORT", "5432"))
DB_NAME = os.getenv("DB_NAME")
DB_USER = os.getenv("DB_USER")
DB_PASS = os.getenv("DB_PASS")

if not all((DB_HOST, DB_NAME, DB_USER, DB_PASS)):
    raise RuntimeError("Configure DB_HOST, DB_NAME, DB_USER e DB_PASS no arquivo python/.env")

# Conexão psycopg2 (escrita: INSERT, UPDATE, DELETE) e SQLAlchemy (leitura: SELECT)
con = pg.connect(host=DB_HOST, port=DB_PORT, dbname=DB_NAME, user=DB_USER, password=DB_PASS)
cnx = URL.create(
    "postgresql+psycopg2",
    username=DB_USER,
    password=DB_PASS,
    host=DB_HOST,
    port=DB_PORT,
    database=DB_NAME,
)
engine = create_engine(cnx)

pn.extension("tabulator", notifications=True)


# 2. Widgets do formulário da tabela LICITACAO (RF06, RN05)
def create_form_widgets():
    return {
        "id_licitacao": pn.widgets.IntInput(
            name="ID da Licitação (para Consultar, Atualizar ou Excluir)",
            value=0,
            start=0,
            sizing_mode="stretch_width",
        ),
        "numero_licitacao": pn.widgets.TextInput(
            name="Número da Licitação * (Único)",
            placeholder="Ex: PE-025/2026",
            sizing_mode="stretch_width",
        ),
        "modalidade": pn.widgets.TextInput(
            name="Modalidade *",
            placeholder="Ex: Pregão Eletrônico, Concorrência Pública...",
            sizing_mode="stretch_width",
        ),
        "data_abertura": pn.widgets.DatePicker(
            name="Data de Abertura *"
        ),
        "situacao": pn.widgets.Select(
            name="Situação da Licitação *",
            options=[
                "aberta",
                "em julgamento",
                "homologada",
                "deserta",
                "cancelada",
            ],
            value="aberta",
            sizing_mode="stretch_width",
        ),
    }


buttonConsultar = pn.widgets.Button(name="🔍 Consultar", button_type="primary")
buttonInserir = pn.widgets.Button(name="➕ Inserir", button_type="success")
buttonAtualizar = pn.widgets.Button(name="✏️ Atualizar", button_type="warning")
buttonExcluir = pn.widgets.Button(name="🗑️ Excluir", button_type="danger")


# 3. Operações CRUD com tratamento de erros e restrições de integridade
def queryAll(mensagem_sucesso=None):
    sql = """
        SELECT id_licitacao, numero_licitacao, modalidade, data_abertura, situacao
        FROM licitacao
        ORDER BY id_licitacao;
    """
    df = pd.read_sql_query(sql, engine)
    tabela = pn.widgets.Tabulator(
        df, show_index=False, sizing_mode="stretch_width", height=400
    )
    if mensagem_sucesso:
        return pn.Column(
            pn.pane.Alert(mensagem_sucesso, alert_type="success"), tabela
        )
    return tabela


def on_consultar(w):
    try:
        id_lic = w["id_licitacao"].value
        num_lic = (w["numero_licitacao"].value or "").strip()

        if id_lic and id_lic > 0:
            sql = text("""
                SELECT id_licitacao, numero_licitacao, modalidade, data_abertura, situacao
                FROM licitacao
                WHERE id_licitacao = :id_lic
            """)
            df = pd.read_sql_query(sql, engine, params={"id_lic": id_lic})
        elif num_lic:
            sql = text("""
                SELECT id_licitacao, numero_licitacao, modalidade, data_abertura, situacao
                FROM licitacao
                WHERE numero_licitacao ILIKE :num_lic
                ORDER BY id_licitacao
            """)
            df = pd.read_sql_query(sql, engine, params={"num_lic": f"%{num_lic}%"})
        else:
            return queryAll()

        if df.empty:
            return pn.pane.Alert(
                "Nenhuma licitação encontrada com os critérios informados.",
                alert_type="warning",
            )
        return pn.widgets.Tabulator(
            df, show_index=False, sizing_mode="stretch_width", height=400
        )
    except Exception as e:
        return pn.pane.Alert(f"Erro na consulta: {str(e)}", alert_type="danger")


def on_inserir(w):
    numero = (w["numero_licitacao"].value or "").strip()
    modalidade = (w["modalidade"].value or "").strip()

    # Validação de campos obrigatórios (RN08)
    if not numero or not modalidade or not w["data_abertura"].value:
        return pn.pane.Alert(
            "Erro de validação (RN08): Preencha Número, Modalidade e Data de Abertura (*).",
            alert_type="danger",
        )
    try:
        with con.cursor() as cursor:
            cursor.execute(
                """
                INSERT INTO licitacao (numero_licitacao, modalidade, data_abertura, situacao)
                VALUES (%s, %s, %s, %s)
                """,
                (
                    numero,
                    modalidade,
                    w["data_abertura"].value,
                    w["situacao"].value,
                ),
            )
        con.commit()
        return queryAll("Licitação cadastrada com sucesso!")
    except Exception as e:
        con.rollback()
        return pn.pane.Alert(
            f"Violação de restrição no banco ao inserir (RN05): {str(e)}",
            alert_type="danger",
        )


def on_atualizar(w):
    id_lic = w["id_licitacao"].value
    numero = (w["numero_licitacao"].value or "").strip()
    modalidade = (w["modalidade"].value or "").strip()

    if not id_lic or id_lic <= 0:
        return pn.pane.Alert(
            "Informe um ID da Licitação válido (> 0) para atualizar.",
            alert_type="warning",
        )
    if not numero or not modalidade:
        return pn.pane.Alert(
            "Erro de validação (RN08): Número da Licitação e Modalidade são obrigatórios.",
            alert_type="danger",
        )
    try:
        with con.cursor() as cursor:
            if w["data_abertura"].value:
                cursor.execute(
                    """
                    UPDATE licitacao
                    SET numero_licitacao = %s,
                        modalidade = %s,
                        data_abertura = %s,
                        situacao = %s
                    WHERE id_licitacao = %s
                    """,
                    (
                        numero,
                        modalidade,
                        w["data_abertura"].value,
                        w["situacao"].value,
                        id_lic,
                    ),
                )
            else:
                cursor.execute(
                    """
                    UPDATE licitacao
                    SET numero_licitacao = %s,
                        modalidade = %s,
                        situacao = %s
                    WHERE id_licitacao = %s
                    """,
                    (
                        numero,
                        modalidade,
                        w["situacao"].value,
                        id_lic,
                    ),
                )
            afetados = cursor.rowcount
        con.commit()
        if afetados == 0:
            return pn.pane.Alert(
                f"Nenhuma licitação encontrada com ID {id_lic} para atualizar.",
                alert_type="warning",
            )
        return queryAll(f"Licitação ID {id_lic} atualizada com sucesso!")
    except Exception as e:
        con.rollback()
        return pn.pane.Alert(
            f"Violação de restrição no banco ao atualizar: {str(e)}",
            alert_type="danger",
        )


def on_excluir(w):
    id_lic = w["id_licitacao"].value
    if not id_lic or id_lic <= 0:
        return pn.pane.Alert(
            "Informe um ID da Licitação válido (> 0) para excluir.",
            alert_type="warning",
        )
    try:
        with con.cursor() as cursor:
            cursor.execute("DELETE FROM licitacao WHERE id_licitacao = %s", (id_lic,))
            afetados = cursor.rowcount
        con.commit()
        if afetados == 0:
            return pn.pane.Alert(
                f"Nenhuma licitação encontrada com ID {id_lic} para excluir.",
                alert_type="warning",
            )
        return queryAll(f"Licitação ID {id_lic} excluída com sucesso!")
    except Exception as e:
        con.rollback()
        return pn.pane.Alert(
            f"Não foi possível excluir: {str(e)}", alert_type="danger"
        )


output_area = pn.Column()


def executar_acao(funcao):
    output_area.clear()
    output_area.append(funcao(form_widgets))


buttonConsultar.on_click(lambda e: executar_acao(on_consultar))
buttonInserir.on_click(lambda e: executar_acao(on_inserir))
buttonAtualizar.on_click(lambda e: executar_acao(on_atualizar))
buttonExcluir.on_click(lambda e: executar_acao(on_excluir))


# 4. Navegação Home / CRUD / Gráficos
main_area = pn.Column()
form_widgets = create_form_widgets()

btn_go_crud = pn.widgets.Button(
    name="📋 Gerenciar Licitações (CRUD)", button_type="primary", width=260
)
btn_go_graficos = pn.widgets.Button(
    name="📊 Licitações por Situação", button_type="success", width=260
)


def create_btn_voltar():
    btn = pn.widgets.Button(name="⬅ Voltar para Home", button_type="default")
    btn.on_click(lambda e: go_to_home())
    return btn


def go_to_home():
    main_area.clear()
    main_area.append(
        pn.Column(
            pn.pane.Markdown("## 🏠 Módulo de Licitações (RF06)"),
            pn.pane.Markdown(
                "Autor: **Yuri Alves de Araújo** | Tabela: `licitacao`"
            ),
            pn.Spacer(height=15),
            btn_go_crud,
            btn_go_graficos,
            align="center",
            margin=(30, 30, 30, 30),
        )
    )


def go_to_crud():
    main_area.clear()
    output_area.clear()
    output_area.append(queryAll())
    main_area.append(
        pn.Row(
            pn.Column(
                pn.pane.Markdown("### 📑 Cadastro e Manutenção de Licitações"),
                form_widgets["id_licitacao"],
                form_widgets["numero_licitacao"],
                form_widgets["modalidade"],
                form_widgets["data_abertura"],
                form_widgets["situacao"],
                pn.Row(buttonConsultar, buttonInserir),
                pn.Row(buttonAtualizar, buttonExcluir),
                pn.Spacer(height=10),
                create_btn_voltar(),
                width=380,
                margin=(10, 20, 10, 10),
            ),
            pn.Column(output_area, sizing_mode="stretch_both"),
        )
    )


def grafico_licitacoes_por_situacao():
    df = pd.read_sql_query(
        "SELECT situacao, COUNT(*) AS quantidade FROM licitacao GROUP BY situacao",
        engine,
    )
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.bar(df["situacao"], df["quantidade"], color=["#1f77b4", "#ff7f0e", "#2ca02c"])
    ax.set_title("Quantidade de Licitações por Situação")
    ax.set_ylabel("Total de Licitações")
    pane = pn.pane.Matplotlib(fig, tight=True)
    plt.close(fig)
    return pane


def go_to_graficos():
    main_area.clear()
    main_area.append(
        pn.Column(
            pn.pane.Markdown("### 📊 Estatísticas de Licitações"),
            grafico_licitacoes_por_situacao(),
            create_btn_voltar(),
            margin=(20, 20, 20, 20),
        )
    )


btn_go_crud.on_click(lambda e: go_to_crud())
btn_go_graficos.on_click(lambda e: go_to_graficos())

go_to_home()

pn.Column(
    pn.pane.Markdown("# 🏛️ Sistema de Acompanhamento de Obras Públicas"),
    main_area,
).servable()